# Denoising Diffusion Probabilistic Model (DDPM) on MNIST
Implement a DDPM similar to the Jackson-Kang tutorial, train on MNIST, sample new images, and evaluate with SSIM and FID.

In [ ]:
!nvidia-smi
%pip install torchmetrics[image] -q

In [ ]:
from __future__ import annotations
from typing import Callable, Iterable, Optional

import gc
import math
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD
from torch import Tensor
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from torchvision.utils import make_grid

from tqdm import tqdm

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

data_dir = "./data"
results_dir = "./outputs/ddpm4"
os.makedirs(results_dir, exist_ok=True)

img_size = (28, 28, 1)
timestep_embedding_dim = 256
n_layers = 8
hidden_dim = 256
n_timesteps = 1000
beta_minmax = [1e-4, 2e-2]

train_batch_size = 128
inference_batch_size = 64
epochs = 20
lr = 2e-4

eval_num_samples = 10000  # Lower this if you need faster evaluation.
save_model_checkpoints = True
save_loss_curves = True

optimizer_names = ["HN_Adam", "Adam", "AMSGrad", "SGD"]

hidden_dims = [hidden_dim for _ in range(n_layers)]

In [ ]:
class HNAdam(Optimizer):
    def __init__(
        self,
        params: Iterable[Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.99),
        eps: float = 1e-8,
        lambda_t0: Optional[float] = None,
    ) -> None:
        if params is None:
            raise ValueError("params cannot be None.")
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if len(betas) != 2:
            raise ValueError("betas must be a tuple of two floats")

        beta1, beta2 = betas
        if not 0.0 <= beta1 < 1.0:
            raise ValueError(f"Invalid beta1 value: {beta1}")
        if not 0.0 <= beta2 < 1.0:
            raise ValueError(f"Invalid beta2 value: {beta2}")

        if lambda_t0 is None:
            lambda_t0 = random.uniform(2.0, 4.0)
        if not 2.0 <= lambda_t0 <= 4.0:
            raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

        defaults = {
            "lr": lr,
            "betas": (beta1, beta2),
            "eps": eps,
            "lambda_t0": lambda_t0,
            "amsgrad": False,
        }
        super().__init__(params, defaults)

        if len(self.param_groups) == 0:
            raise ValueError("optimizer got an empty parameter list")

    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
        loss: Optional[Tensor] = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            lambda_t0: float = group["lambda_t0"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                if grad.is_sparse:
                    raise RuntimeError("HNAdam does not support sparse gradients")

                state = self.state[param]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

                m_prev: Tensor = state["m"]
                v_prev: Tensor = state["v"]
                vhat_prev: Tensor = state["vhat"]

                g_t = grad

                m_t = beta1 * m_prev + (1.0 - beta1) * g_t

                g_abs = g_t.abs()
                m_prev_norm = torch.linalg.vector_norm(m_prev)
                g_abs_norm = torch.linalg.vector_norm(g_abs)
                m_max = torch.maximum(m_prev_norm, g_abs_norm)

                zero = torch.zeros((), dtype=param.dtype, device=param.device)
                ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
                lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

                v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

                if bool((lambda_t < 2.0).item()):
                    group["amsgrad"] = True

                    vhat_t = torch.maximum(vhat_prev, v_t.abs())
                    state["vhat"] = vhat_t

                    denom = vhat_t.pow(1.0 / lambda_t) + eps
                else:
                    group["amsgrad"] = False

                    denom = v_t.pow(1.0 / lambda_t) + eps

                param.addcdiv_(m_t, denom, value=-lr)

                state["m"] = m_t
                state["v"] = v_t

        return loss

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

num_workers = 0
pin_memory = device.type == "cuda"

train_dataset = MNIST(data_dir, transform=transform, train=True, download=True)
test_dataset = MNIST(data_dir, transform=transform, train=False, download=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True,
    )
test_loader = DataLoader(
    test_dataset,
    batch_size=inference_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    )

## Model and diffusion process
The model mirrors the tutorial: a stacked convolutional denoiser with sinusoidal timestep embeddings and a Gaussian diffusion process.

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class ConvBlock(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        activation_fn=None,
        drop_rate=0.0,
        stride=1,
        padding="same",
        dilation=1,
        groups=1,
        bias=True,
        gn=False,
        gn_groups=8,
    ):
        if padding == "same":
            padding = kernel_size // 2 * dilation

        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=groups,
            bias=bias,
        )

        self.activation_fn = nn.SiLU() if activation_fn else None
        self.group_norm = nn.GroupNorm(gn_groups, out_channels) if gn else None

    def forward(self, x, time_embedding=None, residual=False):
        if residual:
            x = x + time_embedding
            y = x
            x = super().forward(x)
            y = y + x
        else:
            y = super().forward(x)
        y = self.group_norm(y) if self.group_norm is not None else y
        y = self.activation_fn(y) if self.activation_fn is not None else y
        return y


class Denoiser(nn.Module):
    def __init__(
        self,
        image_resolution,
        hidden_dims=None,
        diffusion_time_embedding_dim=256,
        n_times=1000,
    ):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [256, 256]

        _, _, img_C = image_resolution
        self.time_embedding = SinusoidalPosEmb(diffusion_time_embedding_dim)

        self.in_project = ConvBlock(img_C, hidden_dims[0], kernel_size=7)
        self.time_project = nn.Sequential(
            ConvBlock(diffusion_time_embedding_dim, hidden_dims[0], kernel_size=1, activation_fn=True),
            ConvBlock(hidden_dims[0], hidden_dims[0], kernel_size=1),
        )

        self.convs = nn.ModuleList(
            [ConvBlock(in_channels=hidden_dims[0], out_channels=hidden_dims[0], kernel_size=3)]
        )
        for idx in range(1, len(hidden_dims)):
            self.convs.append(
                ConvBlock(
                    hidden_dims[idx - 1],
                    hidden_dims[idx],
                    kernel_size=3,
                    dilation=3 ** ((idx - 1) // 2),
                    activation_fn=True,
                    gn=True,
                    gn_groups=8,
                )
            )

        self.out_project = ConvBlock(hidden_dims[-1], out_channels=img_C, kernel_size=3)

    def forward(self, perturbed_x, diffusion_timestep):
        y = perturbed_x
        diffusion_embedding = self.time_embedding(diffusion_timestep)
        diffusion_embedding = self.time_project(diffusion_embedding.unsqueeze(-1).unsqueeze(-2))
        y = self.in_project(y)
        for conv in self.convs:
            y = conv(y, diffusion_embedding, residual=True)
        y = self.out_project(y)
        return y

In [ ]:
class Diffusion(nn.Module):
    def __init__(self, model, image_resolution, n_times=1000, beta_minmax=None, device="cuda"):
        super().__init__()
        if beta_minmax is None:
            beta_minmax = [1e-4, 2e-2]

        self.n_times = n_times
        self.img_H, self.img_W, self.img_C = image_resolution
        self.model = model
        self.device = device

        beta_1, beta_T = beta_minmax
        betas = torch.linspace(start=beta_1, end=beta_T, steps=n_times, device=device)
        self.sqrt_betas = torch.sqrt(betas)

        self.alphas = 1 - betas
        self.sqrt_alphas = torch.sqrt(self.alphas)
        alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - alpha_bars)
        self.sqrt_alpha_bars = torch.sqrt(alpha_bars)

    def extract(self, a, t, x_shape):
        b, *_ = t.shape
        out = a.gather(-1, t)
        return out.reshape(b, *((1,) * (len(x_shape) - 1)))

    def scale_to_minus_one_to_one(self, x):
        return x * 2 - 1

    def reverse_scale_to_zero_to_one(self, x):
        return (x + 1) * 0.5

    def make_noisy(self, x_zeros, t):
        epsilon = torch.randn_like(x_zeros, device=self.device)
        sqrt_alpha_bar = self.extract(self.sqrt_alpha_bars, t, x_zeros.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, t, x_zeros.shape)
        noisy_sample = x_zeros * sqrt_alpha_bar + epsilon * sqrt_one_minus_alpha_bar
        return noisy_sample.detach(), epsilon

    def forward(self, x_zeros):
        x_zeros = self.scale_to_minus_one_to_one(x_zeros)
        b, _, _, _ = x_zeros.shape
        t = torch.randint(low=0, high=self.n_times, size=(b,), device=self.device).long()
        perturbed_images, epsilon = self.make_noisy(x_zeros, t)
        pred_epsilon = self.model(perturbed_images, t)
        return perturbed_images, epsilon, pred_epsilon

    def denoise_at_t(self, x_t, timestep, t):
        if t > 1:
            z = torch.randn_like(x_t, device=self.device)
        else:
            z = torch.zeros_like(x_t, device=self.device)

        epsilon_pred = self.model(x_t, timestep)
        alpha = self.extract(self.alphas, timestep, x_t.shape)
        sqrt_alpha = self.extract(self.sqrt_alphas, timestep, x_t.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, timestep, x_t.shape)
        sqrt_beta = self.extract(self.sqrt_betas, timestep, x_t.shape)
        x_t_minus_1 = (
            1 / sqrt_alpha
            * (x_t - (1 - alpha) / sqrt_one_minus_alpha_bar * epsilon_pred)
            + sqrt_beta * z
        )
        return x_t_minus_1.clamp(-1.0, 1.0)

    def sample(self, N):
        x_t = torch.randn((N, self.img_C, self.img_H, self.img_W), device=self.device)
        for t in range(self.n_times - 1, -1, -1):
            timestep = torch.full((N,), t, device=self.device, dtype=torch.long)
            x_t = self.denoise_at_t(x_t, timestep, t)
        x_0 = self.reverse_scale_to_zero_to_one(x_t)
        return x_0

    def sample_with_intermediates(self, N, num_steps=8):
        x_t = torch.randn((N, self.img_C, self.img_H, self.img_W), device=self.device)
        intermediates = []
        capture_steps = np.linspace(self.n_times - 1, 0, num_steps, dtype=int).tolist()
        for t in range(self.n_times - 1, -1, -1):
            timestep = torch.full((N,), t, device=self.device, dtype=torch.long)
            x_t = self.denoise_at_t(x_t, timestep, t)
            if t in capture_steps:
                intermediates.append(self.reverse_scale_to_zero_to_one(x_t.detach().clone()))
        return intermediates

In [ ]:
def set_seed(seed_value: int) -> None:
    torch.manual_seed(seed_value)
    np.random.seed(seed_value)
    random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)


def build_diffusion() -> Diffusion:
    model = Denoiser(
        image_resolution=img_size,
        hidden_dims=hidden_dims,
        diffusion_time_embedding_dim=timestep_embedding_dim,
        n_times=n_timesteps,
    ).to(device)
    diffusion = Diffusion(
        model,
        image_resolution=img_size,
        n_times=n_timesteps,
        beta_minmax=beta_minmax,
        device=device,
    ).to(device)
    return diffusion


def build_optimizer(name: str, params: Iterable[torch.nn.Parameter]) -> Optimizer:
    if name == "HN_Adam":
        return HNAdam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, lambda_t0=3.0)
    if name == "Adam":
        return Adam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8)
    if name == "AMSGrad":
        return Adam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, amsgrad=True)
    if name == "SGD":
        return SGD(params, lr=lr)
    raise ValueError(f"Unknown optimizer: {name}")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


denoising_loss = nn.MSELoss()

## Training
The model predicts the noise added at timestep $t$ and is trained with MSE loss.

In [ ]:
def collect_real_images(loader: DataLoader, num_samples: int) -> torch.Tensor:
    images = []
    total = 0
    for x, _ in loader:
        images.append(x)
        total += x.size(0)
        if total >= num_samples:
            break
    return torch.cat(images, dim=0)[:num_samples]


def preprocess_for_fid(x: torch.Tensor) -> torch.Tensor:
    if x.size(1) == 1:
        x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
    x = (x * 255).clamp(0, 255).to(torch.uint8)
    return x


def compute_metrics(diffusion: Diffusion, real_images: torch.Tensor) -> tuple[float, float]:
    diffusion.eval()
    fake_images = []
    with torch.no_grad():
        for i in range(0, real_images.size(0), inference_batch_size):
            cur_batch = min(inference_batch_size, real_images.size(0) - i)
            fake_batch = diffusion.sample(N=cur_batch)
            fake_images.append(fake_batch.cpu())
    fake_images = torch.cat(fake_images, dim=0)

    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    for i in range(0, real_images.size(0), inference_batch_size):
        real_batch = real_images[i : i + inference_batch_size].to(device)
        fake_batch = fake_images[i : i + inference_batch_size].to(device)
        ssim_metric.update(fake_batch, real_batch)
    ssim_score = float(ssim_metric.compute().cpu().item())
    ssim_metric.reset()

    fid = FrechetInceptionDistance(feature=2048).to(device)

    def update_fid(images: torch.Tensor, real: bool) -> None:
        for i in range(0, images.size(0), inference_batch_size):
            batch = images[i : i + inference_batch_size].to(device)
            batch = preprocess_for_fid(batch)
            fid.update(batch, real=real)

    update_fid(real_images, real=True)
    update_fid(fake_images, real=False)
    fid_score = float(fid.compute().cpu().item())
    fid.reset()

    return ssim_score, fid_score


eval_samples = min(eval_num_samples, len(test_dataset))
real_images = collect_real_images(test_loader, eval_samples)
perm = torch.randperm(real_images.size(0))
real_images = real_images[perm]

results_records = []
loss_histories = {}
best_diffusion = None
best_optimizer = None
best_loss_overall = float("inf")

loss_dir = os.path.join(results_dir, "loss_curves")
os.makedirs(loss_dir, exist_ok=True)

param_count = None
for name in optimizer_names:
    set_seed(seed)
    diffusion = build_diffusion()
    if param_count is None:
        param_count = count_parameters(diffusion)
        print("Number of model parameters:", param_count)
    optimizer = build_optimizer(name, diffusion.parameters())

    epoch_losses = []
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    for epoch in range(epochs):
        diffusion.train()
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"{name} | Epoch {epoch + 1}/{epochs}")
        for x, _ in progress:
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)
            _, epsilon, pred_epsilon = diffusion(x)
            loss = denoising_loss(pred_epsilon, epsilon)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            progress.set_postfix(loss=loss.item())
        avg_loss = running_loss / len(train_loader)
        epoch_losses.append(avg_loss)
        print(f"{name} | Epoch {epoch + 1}: loss={avg_loss:.6f}")

    if device.type == "cuda":
        torch.cuda.synchronize()
    training_time_s = time.perf_counter() - start_time
    best_loss = float(np.min(epoch_losses))

    ssim_score, fid_score = compute_metrics(diffusion, real_images)

    results_records.append(
        {
            "Optimizer": name,
            "Best Loss (MSE)": best_loss,
            "Training Time (s)": training_time_s,
            "SSIM": ssim_score,
            "FID": fid_score,
        }
    )
    loss_histories[name] = epoch_losses

    if save_loss_curves:
        loss_path = os.path.join(loss_dir, f"loss_{name.lower()}.csv")
        pd.DataFrame({"epoch": range(1, len(epoch_losses) + 1), "loss": epoch_losses}).to_csv(
            loss_path, index=False
        )
    if save_model_checkpoints:
        model_path = os.path.join(results_dir, f"ddpm_{name.lower()}.pt")
        torch.save(diffusion.state_dict(), model_path)

    if best_loss < best_loss_overall:
        best_loss_overall = best_loss
        best_diffusion = diffusion
        best_optimizer = name
    else:
        del diffusion
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results_records)
results_path = os.path.join(results_dir, "optimizer_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved results to {results_path}")
print(f"Best optimizer by loss: {best_optimizer} (loss={best_loss_overall:.6f})")

In [ ]:
plt.figure(figsize=(7, 4))
for name, losses in loss_histories.items():
    plt.plot(range(1, len(losses) + 1), losses, marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Noise Prediction Loss (MSE)")
plt.title("DDPM Training Loss by Optimizer")
plt.grid(True)
plt.legend()
plt.show()

## Sampling
Generate new images by reversing the diffusion process from pure noise.

In [ ]:
if best_diffusion is None:
    raise RuntimeError("Run the training cell before sampling.")

best_diffusion.eval()
with torch.no_grad():
    generated_images = best_diffusion.sample(N=inference_batch_size)

grid = make_grid(generated_images, nrow=8, padding=2, normalize=True)
plt.figure(figsize=(6, 6))
plt.axis("off")
title = f"DDPM Samples ({best_optimizer})" if best_optimizer else "DDPM Samples"
plt.title(title)
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
plt.show()

In [ ]:
if best_diffusion is None:
    raise RuntimeError("Run the training cell before sampling.")

with torch.no_grad():
    intermediates = best_diffusion.sample_with_intermediates(N=1, num_steps=8)

fig, axes = plt.subplots(1, len(intermediates), figsize=(12, 2))
for ax, img in zip(axes, intermediates):
    ax.imshow(img[0].squeeze(0).cpu().numpy(), cmap="gray")
    ax.axis("off")
plt.suptitle("Reverse diffusion steps")
plt.show()

## Results summary
Metrics are computed per optimizer during training and saved to disk.

In [ ]:
print(f"Results saved to {results_path}")
results_df

In [ ]:
table1 = results_df.set_index("Optimizer")
table1